# Well log changepoint detection example

Online changepoint detection on the `well_log` dataset from the Turing Change Point Dataset (TCPD) (Ó Ruanaidh and Fitzgerald, 1996; Van den Burg and Williams, 2020). This notebook shows how to obtain the data, import and define the score and detector, and run it on the data, including resetting upon finding a changepoint.

## Download and load data and annotations

Run the following code to download and import the data from the TCPD github repository.

In [ ]:
!git clone --depth 1 https://github.com/alan-turing-institute/TCPD.git

In [ ]:
import matplotlib.pyplot as plt
import json
import numpy as np

# Update the data directory if it is wrong
DATA_DIR = "TCPD"

# Load data and annotations
with open(f"{DATA_DIR}/datasets/well_log/well_log.json") as f:
    d = json.load(f)
data = np.array(d["series"][0]["raw"], dtype=float)

with open(f"{DATA_DIR}/annotations.json") as f:
    annotations = json.load(f)
annotators = annotations["well_log"]

## Code from the paper

In [ ]:
from gridcp import GridDetector
from gridcp.scores import GaussianMean

score = GaussianMean()
detector = GridDetector(score, threshold = 2.8)

state = detector.init_state()
alarms = []
scores = []
for i, y in enumerate(data):
    state, output = detector.update(state, y)
    scores.append(output["max_score"])
    if output["alarm"]:
        alarms.append(i)
        state = detector.init_state()

## Plot
Upper panel: Data with alarm times and annotated changepoints. Bottom panel: The penalized GaussianMean score with threshold.

In [ ]:
# Plot data and alarm times in top display, penalized score and threshold in the bottom display
colors = plt.cm.tab10.colors
thr = float(detector.threshold[0])

fig2, axes2 = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
# Top display: raw data with annotators marked by triangles, and alarm times
ax = axes2[0]
ax.plot(data, color="steelblue", linewidth=0.8)
for j, a in enumerate(alarms):
    ax.axvline(
        a,
        color="tomato",
        linestyle="-",
        linewidth=1,
        alpha=0.8,
        label="Alarm times" if j == 0 else None,
    )
cp_stack = {}
for k, (ann_id, cps) in enumerate(annotators.items()):
    for cp in cps:
        level = cp_stack.get(cp, 0)
        ax.scatter(
            cp,
            0.03 + level * 0.025,
            transform=ax.get_xaxis_transform(),
            s=12,
            marker="^",
            color=colors[k],
            zorder=3,
        )
        cp_stack[cp] = level + 1

ann_handle = plt.Line2D(
    [],
    [],
    marker="^",
    linestyle="none",
    markersize=5,
    color="grey",
    label="Annotated changepoints",
)
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles=handles + [ann_handle],
    labels=labels + ["Annotated changepoints"],
    loc="upper left",
    fontsize=9,
)
ax.set_ylabel("NMR measurement")

# Bottom display: running max score with threshold
ax = axes2[1]
ax.plot(scores, color="steelblue", linewidth=0.8)
ax.axhline(
    thr,
    color="lightgreen",
    linestyle="--",
    linewidth=1.5,
    label=f"Threshold ({thr:.2f})",
)
ax.set_ylabel("Penalized score")
ax.set_xlabel("Sample index")
ax.legend(loc="upper left", fontsize=9)

plt.suptitle(
    "Alarm times of the GaussianMean detector on the well log dataset",
    fontsize=12,
)
plt.tight_layout()
plt.show()